In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

# 1. Download 10 years of NIFTY 50 data
print("Downloading data...")
df = yf.download('^NSEI', start='2014-01-01', end='2024-01-01')

# Handle multi-level columns (newer yfinance)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Use 'Close' directly; fall back to 'Adj Close' if present
if 'Adj Close' in df.columns:
    df['Close'] = df['Adj Close']
else:
    df['Close'] = df['Close']

print("Columns available:", df.columns.tolist())
print(df.head())

# 2. Engineer features
print("Engineering features...")
df['Target'] = np.log(df['Close'] / df['Close'].shift(1)).shift(-1)
df['Volatility'] = df['Target'].rolling(14).std() * np.sqrt(252)
df['Log_Return_1d'] = np.log(df['Close'] / df['Close'].shift(1))
df['Log_Return_5d'] = np.log(df['Close'] / df['Close'].shift(5))

# RSI (14-day)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# 3. Clean missing values and save to Parquet
df.dropna(inplace=True)
os.makedirs('../data', exist_ok=True)
df.to_parquet('../data/nifty_features.parquet')

print("✅ Data saved successfully to data/nifty_features.parquet")
print(f"Total rows: {len(df)}")
print(df[['Close', 'Target', 'Volatility', 'RSI']].head())

[*********************100%***********************]  1 of 1 completed

Columns available: ['Close', 'High', 'Low', 'Open', 'Volume']
Price             Close         High          Low         Open  Volume
Date                                                                  
2014-01-02  6221.149902  6358.299805  6211.299805  6301.250000  158100
2014-01-03  6211.149902  6221.700195  6171.250000  6194.549805  139000
2014-01-06  6191.450195  6224.700195  6170.250000  6220.850098  118300
2014-01-07  6162.250000  6221.500000  6144.750000  6203.899902  138600
2014-01-08  6174.600098  6192.100098  6160.350098  6178.049805  146900
Engineering features...


✅ Data saved successfully to data/nifty_features.parquet
Total rows: 2439
Price             Close    Target  Volatility        RSI
Date                                                    
2014-01-21  6313.799805  0.003975    0.108699  61.487913
2014-01-22  6338.950195  0.001056    0.107882  63.748870
2014-01-23  6345.649902 -0.012512    0.122109  65.819822
2014-01-24  6266.750000 -0.021109    0.152617  57.774090
2014-01-27  6135.850098 -0.001566    0.152321  47.747447
